<a href="https://colab.research.google.com/github/manomaya10/1WN24CS153-BIS-LAB/blob/main/cuckoo_serach_algo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import numpy as np
import math

jobs = [4, 6, 3, 5, 2, 7]

num_machines = 3

n = 20
Pa = 0.25
MaxIt = 100
alpha = 0.1
lam = 1.5

num_jobs = len(jobs)

def levy_flight(lam):

    sigma = (
        math.gamma(1 + lam) *
        math.sin(math.pi * lam / 2) /
        (
            math.gamma((1 + lam) / 2) *
            lam *
            2 ** ((lam - 1) / 2)
        )
    ) ** (1 / lam)

    u = np.random.normal(0, sigma, num_jobs)
    v = np.random.normal(0, 1, num_jobs)

    step = u / (np.abs(v) ** (1 / lam))

    return step


def decode(solution):

    order = np.argsort(solution)

    machine_load = [0] * num_machines
    schedule = [[] for _ in range(num_machines)]

    for job_index in order:

        machine = np.argmin(machine_load)

        schedule[machine].append(job_index + 1)

        machine_load[machine] += jobs[job_index]

    return schedule, machine_load


def fitness(solution):

    schedule, loads = decode(solution)

    makespan = max(loads)

    return 1 / (1 + makespan)

nests = np.random.rand(n, num_jobs)

scores = np.array([fitness(x) for x in nests])

best_index = np.argmax(scores)
best_nest = nests[best_index].copy()
best_score = scores[best_index]

for t in range(MaxIt):

    for i in range(n):

        step = levy_flight(lam)

        new_solution = nests[i] + alpha * step

        new_solution = np.clip(new_solution, 0, 1)

        new_score = fitness(new_solution)

        j = np.random.randint(n)

        if new_score > scores[j]:

            nests[j] = new_solution
            scores[j] = new_score

    abandon_count = int(Pa * n)

    worst_indices = np.argsort(scores)[:abandon_count]

    for j in worst_indices:

        nests[j] = np.random.rand(num_jobs)
        scores[j] = fitness(nests[j])

    current_best = np.argmax(scores)

    if scores[current_best] > best_score:

        best_score = scores[current_best]
        best_nest = nests[current_best].copy()

best_schedule, loads = decode(best_nest)

print("Best Job Schedule")

for i in range(num_machines):

    print("Machine", i + 1,
          "Jobs:", best_schedule[i],
          "Total Time:", loads[i])

print("Minimum Makespan:", max(loads))

print("Best Fitness:", best_score)


Best Job Schedule
Machine 1 Jobs: [np.int64(4), np.int64(1)] Total Time: 9
Machine 2 Jobs: [np.int64(2), np.int64(3)] Total Time: 9
Machine 3 Jobs: [np.int64(5), np.int64(6)] Total Time: 9
Minimum Makespan: 9
Best Fitness: 0.1
